In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# Move to the working directory, clone the repo, and enter it
%cd /kaggle/working
!git clone https://github.com/sanghyun-son/EDSR-PyTorch
%cd EDSR-PyTorch/src

/kaggle/working
Cloning into 'EDSR-PyTorch'...
remote: Enumerating objects: 806, done.
remote: Total 806 (delta 0), reused 0 (delta 0), pack-reused 806 (from 1)
Receiving objects: 100% (806/806), 63.09 MiB | 38.78 MiB/s, done.
Resolving deltas: 100% (516/516), done.
/kaggle/working/EDSR-PyTorch/src


In [2]:
import os
import glob
import shutil

# 0. WIPE THE SLATE CLEAN (Removes old attempts)
dataset_root = '/kaggle/working/perfect_dataset'
if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)

# 1. Set up the exact DIV2K directories
base_target = os.path.join(dataset_root, 'DIV2K')
hr_target = os.path.join(base_target, 'DIV2K_train_HR')
lr_target = os.path.join(base_target, 'DIV2K_train_LR_bicubic/X4')

os.makedirs(hr_target, exist_ok=True)
os.makedirs(lr_target, exist_ok=True)

# 2. EXACT paths from your Kaggle Radar
hr_src_folder = '/kaggle/input/datasets/marieclaudsies/edsr-dataset/EDSR_Dataset/HR'
lr_src_folder = '/kaggle/input/datasets/marieclaudsies/edsr-dataset/EDSR_Dataset/LR/X4'

hr_images = sorted(glob.glob(os.path.join(hr_src_folder, '*.png')))
lr_images = sorted(glob.glob(os.path.join(lr_src_folder, '*.png')))

print(f"Found {len(hr_images)} HR images and {len(lr_images)} LR images.")

# 3. Copy and forcefully rename them
if len(hr_images) > 0 and len(hr_images) == len(lr_images):
    print("Copying and renaming 13,000+ files... give this a minute...")
    for i, (hr_path, lr_path) in enumerate(zip(hr_images, lr_images)):
        # EDSR demands strict integer numbering
        new_name = f"{i+1:04d}.png" 
        # It ALSO hardcodes an 'x4' suffix for the Low-Res files
        new_name_lr = f"{i+1:04d}x4.png" 
        
        shutil.copy(hr_path, os.path.join(hr_target, new_name))
        shutil.copy(lr_path, os.path.join(lr_target, new_name_lr))
        
    print("✅ Files successfully copied and strictly renamed! The dataloader will accept these.")
else:
    print("❌ Error: Mismatch in number of images or no images found.")

Found 13390 HR images and 13390 LR images.
Copying and renaming 13,000+ files... give this a minute...
✅ Files successfully copied and strictly renamed! The dataloader will accept these.


In [4]:
!python main.py \
    --model EDSR \
    --scale 4 \
    --dir_data /kaggle/working/perfect_dataset \
    --data_train DIV2K \
    --ext img \
    --pre_train download \
    --epochs 30 \
    --test_every 10 \
    --n_threads 4 \
    --save edsr_results_experiment

Making model...
Download the model
Downloading: "https://cv.snu.ac.kr/research/EDSR/models/edsr_baseline_x4-6b446fab.pt" to ../models/edsr_baseline_x4-6b446fab.pt
100%|███████████████████████████████████████| 5.80M/5.80M [00:51<00:00, 117kB/s]
Preparing loss function:
1.000 * L1
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:578: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 1]	Learning rate: 1.00e-4

Evaluation:
100%|███████████████████████████████████████████| 10/10 [00:00<00:00, 38.27it/s]
[DIV2K x4]	PSNR: 24.577 (Best: 24.577 @epoch 1)
Forward: 0.27s

Saving...
Total: 1.47s

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:578: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 2]	Learning rate: 1.00e-4

Evaluation:
100%|████████████████████████████████

In [8]:
import os

print("--- KAGGLE FILE RADAR ---")
for root, dirs, files in os.walk('/kaggle/input'):
    if files:
        print(f"📁 Found {len(files)} files in: {root}")
        print(f"   📄 Example file: {files[0]}")

--- KAGGLE FILE RADAR ---
📁 Found 13390 files in: /kaggle/input/datasets/marieclaudsies/edsr-dataset/EDSR_Dataset/LR/X4
   📄 Example file: img1_2226.png
📁 Found 13390 files in: /kaggle/input/datasets/marieclaudsies/edsr-dataset/EDSR_Dataset/HR
   📄 Example file: img1_2226.png


### test the results

In [ ]:
### Step 1: Extract the "Final Exam" Patches
#### Run this in a new cell. It will hunt down the new vv file, slice out 5 distinct high-resolution patches, shrink them to create the low-resolution inputs, and save them in a fresh test folder.

In [7]:
import os
import glob
from PIL import Image

# Prevent decompression bomb errors for large TIFFs
Image.MAX_IMAGE_PIXELS = None 

# 1. Create dedicated test folders
hr_test_dir = '/kaggle/working/sar_test_images/HR'
lr_test_dir = '/kaggle/working/sar_test_images/LR'
os.makedirs(hr_test_dir, exist_ok=True)
os.makedirs(lr_test_dir, exist_ok=True)

# 2. Automatically find the new 'vv' tiff file
print("Hunting for the VV polarization file...")
# Using recursive glob to bypass any hidden Kaggle folder nesting
tiff_files = glob.glob('/kaggle/input/**/*vv*', recursive=True)

if not tiff_files:
    print("❌ Could not find the file. Check if it has a different extension.")
else:
    test_tiff_path = tiff_files[0]
    print(f"✅ Found test image: {test_tiff_path}")
    
    # 3. Process the Image
    try:
        img = Image.open(test_tiff_path).convert('RGB')
        width, height = img.size
        
        count = 0
        patch_size = 256
        scale = 4
        
        # Start in the middle of the image to avoid empty black borders
        start_y = height // 2
        start_x = width // 2
        
        print("Slicing patches...")
        for i in range(5):
            # Shift the x-axis for each patch so we get 5 different areas
            x = start_x + (i * patch_size)
            y = start_y
            
            box = (x, y, x + patch_size, y + patch_size)
            hr_patch = img.crop(box)
            
            # Save Ground Truth (HR)
            patch_name = f"test_patch_{count:02d}.png"
            hr_patch.save(os.path.join(hr_test_dir, patch_name))
            
            # Create and Save Input (LR) using Bicubic shrinking
            resample_method = getattr(Image, 'Resampling', Image).BICUBIC
            lr_patch = hr_patch.resize((patch_size // scale, patch_size // scale), resample_method)
            lr_patch.save(os.path.join(lr_test_dir, patch_name))
            
            count += 1
            
        print(f"✅ Extracted {count} unseen test patches!")
        print(f"HR Ground Truth saved to: {hr_test_dir}")
        print(f"LR Inputs saved to: {lr_test_dir}")

    except Exception as e:
        print(f"❌ Error processing TIFF: {e}")

Hunting for the VV polarization file...
✅ Found test image: /kaggle/input/datasets/marieclaudsies/test-sar-dataset/s1a-iw-grd-vv-20260317t164324-20260317t164349-063670-0800f4-001.tiff
Slicing patches...
✅ Extracted 5 unseen test patches!
HR Ground Truth saved to: /kaggle/working/sar_test_images/HR
LR Inputs saved to: /kaggle/working/sar_test_images/LR


### Step 1: Run the Inference
#### Run this command in a Jupyter cell to feed those 5 Low-Res patches into your EDSR model.

In [8]:
%cd /kaggle/working/EDSR-PyTorch/src

!python main.py \
    --model EDSR \
    --scale 4 \
    --test_only \
    --pre_train /kaggle/working/EDSR-PyTorch/experiment/edsr_results_experiment/model/model_best.pt \
    --dir_demo /kaggle/working/sar_test_images/LR \
    --data_test Demo \
    --save_results

/kaggle/working/EDSR-PyTorch/src
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Making model...
Load the model from /kaggle/working/EDSR-PyTorch/experiment/edsr_results_experiment/model/model_best.pt

Evaluation:
  0%|                                                     | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive 

### Step 3: The Automated Grader
#### Once the inference is done, run this Python script. It will automatically match EDSR's generated images against the High-Resolution Ground Truths, calculate the PSNR and SSIM for each, and give you the final average score.

In [12]:
import os
import glob
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim

# Paths to the model's answers and the true answer key
results_dir = '/kaggle/working/EDSR-PyTorch/experiment/test/results-Demo/'
hr_dir = '/kaggle/working/sar_test_images/HR/'

# Find all EDSR generated images using the CORRECTED naming convention
generated_images = sorted(glob.glob(os.path.join(results_dir, '*_x4_SR.png')))

psnr_scores = []
ssim_scores = []

print("📊 --- FINAL EDSR SAR EVALUATION ---")

for gen_path in generated_images:
    # 1. Fix the naming mismatch (test_patch_01_x4_SR.png -> test_patch_01.png)
    base_name = os.path.basename(gen_path).replace('_x4_SR.png', '.png')
    
    # 2. Skip the old optical bird image if it's still in the folder!
    if "0853" in base_name:
        continue
        
    gt_path = os.path.join(hr_dir, base_name)
    
    try:
        img_gen = np.array(Image.open(gen_path).convert('RGB'))
        img_gt = np.array(Image.open(gt_path).convert('RGB'))
        
        # Calculate mathematical accuracy
        psnr_val = compare_psnr(img_gt, img_gen, data_range=255)
        ssim_val = compare_ssim(img_gt, img_gen, data_range=255, channel_axis=-1)
        
        psnr_scores.append(psnr_val)
        ssim_scores.append(ssim_val)
        
        print(f"Patch {base_name} | PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")
        
    except Exception as e:
        print(f"❌ Error grading {base_name}: {e}")

# Calculate the final averages
if psnr_scores:
    print("-" * 35)
    print(f"🏆 AVERAGE PSNR : {np.mean(psnr_scores):.2f} dB")
    print(f"🏆 AVERAGE SSIM : {np.mean(ssim_scores):.4f}")
else:
    print("❌ No valid SAR patches found to grade.")

📊 --- FINAL EDSR SAR EVALUATION ---
Patch test_patch_00.png | PSNR: 17.97 dB | SSIM: 0.5362
Patch test_patch_01.png | PSNR: 17.88 dB | SSIM: 0.4912
Patch test_patch_02.png | PSNR: 17.70 dB | SSIM: 0.4549
Patch test_patch_03.png | PSNR: 18.44 dB | SSIM: 0.4883
Patch test_patch_04.png | PSNR: 18.40 dB | SSIM: 0.4757
-----------------------------------
🏆 AVERAGE PSNR : 18.08 dB
🏆 AVERAGE SSIM : 0.4893
